# 第 19 堂｜C/C++ 陣列與指標（Arrays & Pointers）

**本堂重點：** 陣列的索引與記憶體位置、指標與解參照、指標位移、`sizeof`，以及 `max_element` 的範圍。

**學習目標**

1. 看懂 `a[i]`、`*(a + i)` 與 `*p` 的關係。
2. 分辨「陣列名稱」與「指標變數」，知道哪些操作可以使用 `++`。
3. 正確使用半開區間 `[begin, end)`，避免超出陣列範圍。

> **執行環境：** 此 Notebook 延續原本的 `C_setting` 設定，以 Python kernel 執行 `%%cpp`。執行前須確定環境中已有 `C_setting.py` 或同名模組；每個 `%%cpp` cell 都是獨立的 C++ 程式。

## 0. 載入 C/C++ Notebook 環境

先執行下方儲存格，再由上往下執行 C++ 範例。

In [ ]:
from C_setting import *

print("C / C++ Notebook 環境已載入。")
print("可以開始使用 %%c 與 %%cpp。")


## 1. 陣列（Array）與索引

宣告一個含 5 個整數的陣列：

```cpp
int a[5] = {10, 20, 30, 40, 50};
```

| 索引 `i` | 0 | 1 | 2 | 3 | 4 |
|:--|--:|--:|--:|--:|--:|
| `a[i]` | 10 | 20 | 30 | 40 | 50 |

陣列的索引從 **0** 開始，合法範圍是 `0` 到 `4`。在相同陣列內，元素依序連續存放於記憶體中。

## 2. 陣列名稱、指標與解參照

```cpp
int a[5] = {10, 20, 30, 40, 50};
int *p = a;          // p 指向 a[0]
// int *p = &a[0];  // 與上一行的初始化效果相同
```

- `a` 是**陣列**，在這類運算式中通常會轉換成指向第一個元素的指標。
- `p` 是**指標變數**，儲存某個 `int` 元素的位址。
- `*p` 是**解參照**：取得或修改 `p` 所指位置的值。
- `a[0]` 是整數值，不是位址，因此 `int *p = a[0];` **不是合法的指標初始化**。

### 範例 A：用指標修改陣列元素

將 `p` 指向的第一個元素改為 `40`，再印出 `a[0]`。**預期輸出：`40`。**

In [ ]:
%%cpp

#include <iostream>
using namespace std;

int main() {
    int a[5] = {10, 20, 30, 40, 50};
    int *p = a;

    *p = 40;  // 等同於 a[0] = 40;
    cout << a[0] << endl;

    return 0;
}


## 3. 索引與指標運算的等價關係

對合法索引 `i`：

```cpp
a[0] == *(a + 0)
a[1] == *(a + 1)
a[2] == *(a + 2)
```

上面是**值相等的概念示意**，不是要直接複製成宣告。`a + i` 會移動 **i 個元素**，不是移動 i 個位元組。

### 範例 B：移動指標

`p++` 可以讓指標移到下一個元素；但 `a++` **不能使用**，因為陣列名稱不是能遞增的指標變數。**預期輸出依序為 `10`、`20`。**

In [ ]:
%%cpp

#include <iostream>
using namespace std;

int main() {
    int a[5] = {10, 20, 30, 40, 50};
    int *p = a;

    cout << *p << endl;  // a[0]：10
    ++p;                 // 現在指向 a[1]
    cout << *p << endl;  // a[1]：20

    return 0;
}


## 4. `sizeof(a)` 與 `sizeof(p)` 不同

- `sizeof(a)`：**整個陣列**佔用的位元組數，這裡為 `5 * sizeof(int)`。
- `sizeof(p)`：**指標變數本身**的位元組數。

> 在常見的 64 位元、`int` 為 4 bytes 的環境，結果可能是 `20` 與 `8`；但實際大小依編譯器及平台而定。`sizeof(a) / sizeof(a[0])` 可在此作用域中計算陣列的元素數量。

In [ ]:
%%cpp

#include <iostream>
using namespace std;

int main() {
    int a[5] = {10, 20, 30, 40, 50};
    int *p = a;

    cout << sizeof(a) << endl;  // 整個陣列佔用的 bytes
    cout << sizeof(p) << endl;  // 指標本身佔用的 bytes

    return 0;
}


## 5. 指標範圍與陣列邊界

對長度為 5 的陣列：

```cpp
int a[5] = {10, 20, 30, 40, 50};
int *p = a;

// 合法元素位址：a + 0、a + 1、...、a + 4
// a + 5 可以作為「尾端下一格」的邊界，但不能解參照。
// *(a + 5) 或 a[5]：越界存取，屬於未定義行為。
```

常見寫法是將 `a + 5` 視為**不包含的終點**，迴圈只處理前 5 個元素。

### 範例 C：安全走訪陣列

**預期輸出：** `10 20 30 40 50`。

In [ ]:
%%cpp

#include <iostream>
using namespace std;

int main() {
    int a[5] = {10, 20, 30, 40, 50};

    for (int *p = a; p != a + 5; ++p) {
        cout << *p;
        if (p != a + 4) {
            cout << " ";
        }
    }
    cout << endl;

    return 0;
}


## 6. `max_element`：注意搜尋終點

`std::max_element(begin, end)` 會在 **`[begin, end)`** 範圍內找最大元素（`end` 不包含在內），並回傳指向該元素的迭代器；對陣列而言這裡可視為指標。

- `max_element(a, a + 4)`：只檢查 `a[0]` 到 `a[3]`，**漏掉 `a[4]`**。
- `max_element(a, a + 5)`：檢查全部 5 個元素。
- 使用前記得引入 `#include <algorithm>`。

### 範例 D：找出整個陣列的最大值

**預期輸出：`50`。**

In [ ]:
%%cpp

#include <algorithm>
#include <iostream>
using namespace std;

int main() {
    int a[5] = {10, 20, 30, 40, 50};

    cout << *max_element(a, a + 5) << endl;

    return 0;
}


## 7. 本堂檢核

1. `int *p = a;` 和 `int *p = &a[0];` 的效果有何關係？
2. 為什麼可以 `p++`，卻不能 `a++`？
3. 為什麼 `*(a + 5)` 不安全，但 `a + 5` 能作為迴圈終點？
4. `max_element(a, a + 4)` 和 `max_element(a, a + 5)` 的搜尋範圍差在哪裡？
5. `sizeof(a)` 與 `sizeof(p)` 各自計算什麼？

**重點整理：** 陣列索引與解參照是兩種讀寫元素的方式；但**陣列本身不等於指標變數**。操作位址時要小心範圍，尤其是 `[begin, end)` 的右端點。